# บทเรียนที่ 13 - ความทรงจำของเอเย่นต์กับกราฟความรู้ Cognee


## การตั้งค่า

สมุดบันทึกนี้สาธิตวิธีการสร้าง **ผู้ช่วยโค้ดดิ้งอัจฉริยะ** พร้อมหน่วยความจำถาวรโดยใช้กราฟความรู้ [**Cognee**](https://www.cognee.ai/) และ **Microsoft Agent Framework** (MAF)

Cognee แปลงข้อความที่ไม่มีโครงสร้างเป็นกราฟความรู้ที่มีโครงสร้างและสามารถสืบค้นได้ โดยใช้เวกเตอร์ฝังตัว — เพื่อมอบหน่วยความจำระยะยาวที่มีความสัมพันธ์ลึกซึ้งแก่เอเจนต์ของคุณ

### สิ่งที่คุณจะได้เรียนรู้
1. **สร้างกราฟความรู้**: แปลงโปรไฟล์นักพัฒนาและแนวปฏิบัติที่ดีที่สุดให้เป็นความรู้ที่มีโครงสร้างและสืบค้นได้
2. **รวม Cognee กับ MAF**: ใช้ฟังก์ชัน `@tool` เพื่อให้เอเจนต์ MAF สามารถสืบค้นกราฟความรู้ของ Cognee ได้
3. **บทสนทนาที่รู้บริบทของเซสชัน**: รักษาบริบทในการถามหลายคำถามภายในเซสชันเดียวกัน
4. **หน่วยความจำระยะยาว**: บันทึกความรู้สำคัญข้ามเซสชันและดึงข้อมูลในบทสนทนาใหม่ ๆ

### ข้อกำหนดเบื้องต้น
- Python 3.9+
- Redis ที่รันบนเครื่อง (`docker run -d -p 6379:6379 redis`) สำหรับจัดการเซสชัน
- กุญแจ API ของ LLM (เช่น OpenAI) — ตั้งค่า `LLM_API_KEY` ในไฟล์ `.env`
- `CACHING=true` ในไฟล์ `.env` (จำเป็นสำหรับเซสชัน Cognee)
- โครงการ Azure AI Foundry พร้อมโมเดลแชทที่ถูกปรับใช้งาน
- ตั้งค่า `AZURE_AI_PROJECT_ENDPOINT` และ `AZURE_AI_MODEL_DEPLOYMENT_NAME` ในไฟล์ `.env`
- เข้าสู่ระบบ Azure CLI แล้ว (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ประเภทของหน่วยความจำของเอเจนต์

สมุดบันทึกนี้สำรวจหน่วยความจำสามประเภทเดียวกันจากสมุดบันทึกบทเรียน 13 หลักหลัก แต่ใช้ Cognee เป็นแบ็กเอนด์ของหน่วยความจำระยะยาว:

| ประเภทของหน่วยความจำ | กลไก | อายุการใช้งาน |
|---|---|---|
| **ทำงาน** | `agent.create_session()` (MAF) | การสนทนาครั้งเดียว |
| **ระยะสั้น** | แคชเซสชัน Cognee (Redis) | เซสชันเดียว |
| **ระยะยาว** | กราฟความรู้ Cognee + เวกเตอร์ | ถาวร |

### สถาปัตยกรรมหน่วยความจำของ Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## เตรียมที่เก็บ Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — การสร้างฐานความรู้

เรารับข้อมูลสามประเภทเพื่อสร้างฐานความรู้ที่ครอบคลุมสำหรับผู้ช่วยเขียนโค้ดของเรา:

1. **ข้อมูลโปรไฟล์นักพัฒนา** — ความเชี่ยวชาญส่วนตัวและพื้นหลังทางเทคนิค
2. **แนวทางปฏิบัติที่ดีที่สุดของ Python** — Zen of Python พร้อมแนวทางใช้งานจริง
3. **บทสนทนาในอดีต** — ชุดคำถาม-คำตอบที่ผ่านมา ระหว่างนักพัฒนาและผู้ช่วย AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## แสดงภาพความสัมพันธ์ความรู้

Cognee สามารถเรนเดอร์ภาพแสดงผลแบบ HTML ที่โต้ตอบได้ของเอนทิตีและความสัมพันธ์ที่มันสกัดออกมาได้


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## เพิ่มความทรงจำด้วย Memify

`memify()` วิเคราะห์กราฟความรู้และสร้างกฎเกณฑ์อัจฉริยะ — ระบุรูปแบบ แนวทางปฏิบัติที่ดีที่สุด และความสัมพันธ์ระหว่างแนวคิดต่างๆ


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — ตัวแทน MAF กับ Cognee Tools

ตอนนี้เราสร้างตัวแทน MAF ที่สามารถสอบถามกราฟความรู้ของ Cognee ผ่านฟังก์ชัน `@tool` ฟังก์ชันนี้ช่วยให้ตัวแทนใช้พลังเต็มที่ของการค้นหาทางความหมายที่รับรู้กราฟในขณะที่ยังคงรักษาบริบทการสนทนาไว้ผ่านเซสชันต่าง ๆ ได้


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## หน่วยความจำในการทำงานกับเซสชัน

`AgentSession` (สร้างขึ้นผ่าน `agent.create_session()`) ให้หน่วยความจำในการทำงานภายในเซสชัน ตัวแทนสามารถอ้างอิงกลับไปยังข้อความก่อนหน้าในขณะที่ยังสามารถสอบถามกราฟความรู้ระยะยาวของ Cognee ได้ด้วยเช่นกัน


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## เซสชันใหม่ — หน่วยความจำระยะยาวยังคงอยู่

การเริ่มเซสชันใหม่จะล้างหน่วยความจำการทำงาน แต่กราฟความรู้ Cognee ยังคงสามารถใช้งานได้ ตัวแทนสามารถดึงความรู้ระยะยาวเดียวกันในบทสนทนาใหม่ทั้งหมดได้


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## สรุป

ในโน้ตบุ๊กนี้คุณได้สร้างผู้ช่วยเขียนโค้ดที่รวม **หน่วยความจำชั่วคราวของ MAF** (`agent.create_session()`) กับ **กราฟความรู้ระยะยาวของ Cognee**

### สิ่งที่คุณได้เรียนรู้
1. **การสร้างกราฟความรู้**: Cognee อ่านข้อมูลข้อความที่ไม่มีโครงสร้างและสร้างกราฟร่วมกับหน่วยความจำเวกเตอร์
2. **การเพิ่มความสมบูรณ์ของกราฟด้วย memify**: ข้อเท็จจริงที่ได้มาและความสัมพันธ์ที่ลึกซึ้งขึ้นบนกราฟที่มีอยู่ของคุณ
3. **การผสานรวม MAF + Cognee**: ฟังก์ชัน `@tool` ช่วยให้ตัวแทน MAF สามารถสอบถามกราฟของ Cognee ได้อย่างเป็นธรรมชาติ
4. **หน่วยความจำชั่วคราว + หน่วยความจำระยะยาว**: `AgentSession` (ผ่าน `agent.create_session()`) ให้บริบทของเซสชันขณะที่ Cognee ให้ความรู้ที่ถาวร
5. **การค้นหาแบบกรองด้วย NodeSets**: กำหนดเป้าหมายเฉพาะกลุ่มย่อยของกราฟความรู้ (เช่น เฉพาะหลักการ)

### ประเด็นสำคัญที่ควรจดจำ
- **Cognee** เปลี่ยนข้อความดิบให้เป็นหน่วยความจำที่มีโครงสร้างและรับรู้ความสัมพันธ์ — มีประสิทธิภาพมากกว่าการเก็บเวกเตอร์แบบราบเรียบ
- **ฟังก์ชัน `@tool`** เชื่อมต่อตัวแทน MAF และระบบความรู้ภายนอกได้อย่างสะอาด
- **`AgentSession`** (ผ่าน `agent.create_session()`) แยกบริบทของแต่ละการสนทนาออกจากความรู้ที่เก็บไว้ระยะยาว
- กราฟความรู้เดียวกันนี้ให้บริการหลายเซสชันและหลายตัวแทน

### การใช้งานในโลกจริง
- **ผู้ช่วยนักพัฒนาซอฟต์แวร์**: ตรวจสอบโค้ด วิเคราะห์เหตุการณ์ ผู้ช่วยด้านสถาปัตยกรรม
- **ผู้ช่วยลูกค้า**: ตัวแทนสนับสนุนผ่านเอกสารผลิตภัณฑ์ คำถามที่พบบ่อย และบันทึก CRM
- **ผู้ช่วยผู้เชี่ยวชาญภายใน**: ผู้ช่วยด้านนโยบาย กฎหมาย หรือความปลอดภัยที่วิเคราะห์แนวทาง
- **ชั้นข้อมูลแบบรวมศูนย์**: รวมข้อมูลแบบมีโครงสร้างและไม่มีโครงสร้างเข้าเป็นกราฟที่สามารถสืบค้นได้

### ขั้นตอนถัดไป
- ทดลองการรับรู้เวลาภายใน Cognee
- กำหนดออนโทโลยี OWL สำหรับคุณภาพกราฟในโดเมนเฉพาะ
- เพิ่มระบบตอบรับจากผู้ใช้เพื่อปรับปรุงการดึงข้อมูลตามเวลา
- ขยายไปยังระบบหลายตัวแทนที่ใช้ชั้นหน่วยความจำ Cognee ร่วมกัน


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**คำปฏิเสธความรับผิดชอบ**:  
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้มีความถูกต้อง แต่อย่างไรก็ตาม โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่แม่นยำ เอกสารต้นฉบับในภาษาต้นฉบับควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่มีความสำคัญ แนะนำให้ใช้บริการแปลโดยผู้เชี่ยวชาญด้านภาษามนุษย์ เราไม่มีความรับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดใด ๆ ที่เกิดจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
